# 示例 4：被动性评估和强制执行为了演示矢量拟合类的被动性评估和强制执行功能，我们再次使用环形槽示例中的 2 端口模型。请参阅其他矢量拟合示例笔记本，以获取关于拟合过程的更通用的说明。

In [ ]:
import matplotlib.pyplot as mplt
import numpy as np

import skrf
from skrf.vectorFitting import VectorFitting

# load and fit the ring slot network with 3 poles
nw = skrf.data.ring_slot
vf = VectorFitting(nw)
vf.vector_fit(n_poles_real=3, n_poles_cmplx=0)

# plot fitting results
freqs = np.linspace(0, 200e9, 201)
fig, ax = mplt.subplots(2, 2)
fig.set_size_inches(12, 8)
vf.plot_s_mag(0, 0, freqs=freqs, ax=ax[0][0]) # s11
vf.plot_s_mag(0, 1, freqs=freqs, ax=ax[0][1]) # s12
vf.plot_s_mag(1, 0, freqs=freqs, ax=ax[1][0]) # s21
vf.plot_s_mag(1, 1, freqs=freqs, ax=ax[1][1]) # s22
fig.tight_layout()
mplt.show()

拟合结果看起来不错，但打印了一条关于非被动向量拟合的 UserWarning。 在进一步调查此问题之前，让我们检查一下均方根误差：

In [ ]:
vf.get_rms_error()

通常，小于 0.05 的均方根误差表明拟合效果良好，并证实了我们的光学检查结果。但是，拟合模型的被动性如何呢？

In [ ]:
vf.is_passive()

为什么模型不是无源的？原始的环形槽数据不应该代表一个无源网络吗？

In [ ]:
nw.is_passive()

网络数据是无源的，但拟合的矢量模型是有源的。让我们进一步调查（并纠正？）这个问题。

In [ ]:
# plot singular values of vector fitted scattering matrix
freqs = np.linspace(0, 200e9, 201)
fig, ax = mplt.subplots(1, 1)
fig.set_size_inches(6, 4)
vf.plot_s_singular(freqs=freqs, ax=ax)
fig.tight_layout()
mplt.show()

拟合后的散射矩阵的一个奇异值在某些频率下大于1。这确实表明该模型是非无源的。为了进行进一步分析，您可以获取所有存在无源性问题的频率范围列表：

In [ ]:
vf.passivity_test()

该网络在两个频率范围内不是被动的：从直流到大约 27.8 GHz，以及从 84.3 GHz 到 98.5 GHz。幸运的是，可以通过强制使其满足被动性条件来获得被动矢量拟合模型：

In [ ]:
vf.passivity_enforce()

在强制使其满足无源条件后，网络在所有频率下都应该表现出无源特性。让我们自己检查一下：

In [ ]:
vf.is_passive()

In [ ]:
vf.passivity_test()

In [ ]:
# plot singular values of vector fitted scattering matrix
freqs = np.linspace(0, 200e9, 201)
fig, ax = mplt.subplots(1, 1)
fig.set_size_inches(6, 4)
vf.plot_s_singular(freqs=freqs, ax=ax)
fig.tight_layout()
mplt.show()

好的，该模型现在是无源的。但是，它是否仍然符合原始网络数据？

In [ ]:
# plot fitting results again after passivity enforcement
freqs = np.linspace(0, 200e9, 201)
fig, ax = mplt.subplots(2, 2)
fig.set_size_inches(12, 8)
vf.plot_s_mag(0, 0, freqs=freqs, ax=ax[0][0]) # s11
vf.plot_s_mag(0, 1, freqs=freqs, ax=ax[0][1]) # s12
vf.plot_s_mag(1, 0, freqs=freqs, ax=ax[1][0]) # s21
vf.plot_s_mag(1, 1, freqs=freqs, ax=ax[1][1]) # s22
fig.tight_layout()
mplt.show()

除了目视检查之外，让我们再次检查均方根误差：

In [ ]:
vf.get_rms_error()

是的，该模型仍然很好地拟合原始数据，与上述第一个非被动拟合结果的差异可以忽略不计：均方根误差仍然非常低。